# **Data preprocessing**

In [1]:
!pip install transformers datasets
import json
import pandas as pd
import numpy as np
import transformers
import sklearn
import datasets
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

df_train = pd.read_json('project_training.json')
df_val = pd.read_json('project_validation.json') 

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 46.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 KB 38.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 KB 25.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 95.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 KB 11.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 KB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 KB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 KB 18.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 KB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 KB 14.7 MB/s eta 0:00:00
     ━━━━━━━━━━━

In [2]:
df_train["climate"] = df_train["climate"].replace({"yes": 0, "no": 1})
df_train["sentiment"] = df_train["sentiment"].replace({"opportunity": 0, "neutral": 1, "risk": 2})
df_train["specificity"] = df_train["specificity"].replace({"spec": 0, "non": 1})
df_train["commitment"] = df_train["commitment"].replace({"yes": 0, "no": 1})
df_train

,text,climate,sentiment,commitment,specificity
0,The accelerator programs have sub-portfolios o...,0,1.0,1.0,1.0
1,"Also by means of BNDES Finem, we offer credit ...",1,NaN,NaN,NaN
2,Climate change Climate change exposes UPM to v...,0,2.0,1.0,1.0
3,Several tools and methodologies aimed at asses...,0,2.0,0.0,1.0
4,We worked with the UK government to accelerate...,0,0.0,0.0,0.0
...,...,...,...,...,...
395,"At the beginning of 2019, VINCI Airports signe...",1,NaN,NaN,NaN
396,We have also signed up to the Partnership for ...,0,2.0,0.0,1.0
397,Suzano also is involved and spearheads externa...,0,1.0,0.0,1.0
398,Risks to the Group’s reputation Risks include ...,1,NaN,NaN,NaN


In [3]:
df_val["climate"] = df_val["climate"].replace({"yes": 0, "no": 1})
df_val["sentiment"] = df_val["sentiment"].replace({"opportunity": 0, "neutral": 1, "risk": 2})
df_val["specificity"] = df_val["specificity"].replace({"spec": 0, "non": 1})
df_val["commitment"] = df_val["commitment"].replace({"yes": 0, "no": 1})
df_val

,text,climate,sentiment,commitment,specificity
0,"Except as required by law, the Bank does not u...",1,NaN,NaN,NaN
1,"In terms of thermal coal, the Group has set an...",0,1.0,0.0,0.0
2,The engagement is driving improved conversatio...,0,1.0,0.0,1.0
3,The tool does this by tracking the generation ...,0,1.0,1.0,1.0
4,We also support and advise companies in the La...,0,0.0,0.0,0.0
...,...,...,...,...,...
395,"JetBlue drives safety management in many ways,...",1,NaN,NaN,NaN
396,10 Representative Concentration Pathways (RCP)...,0,1.0,1.0,1.0
397,Building on our history of energy efficiency i...,0,1.0,0.0,0.0
398,"In 2019, PME redesigned the passively managed ...",0,2.0,0.0,0.0


In [4]:
from transformers import pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
from sklearn.metrics import classification_report
from transformers import set_seed
from sklearn.metrics import f1_score
set_seed(42)

In [79]:
checkpoint = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    tokens = tokenizer(batch['text'], truncation=True, max_length=512) # finetune: 10,50,100,512
    tokens['label'] = batch['label']
    return tokens

Could not locate the tokenizer configuration file, will try to use the model config instead.
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading file vocab.json from cache at /root/.cache/hugging

# **The first approach: 4 models for 4 labels**

# *1st approach, 1st model*



In [80]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [81]:
df_train_model1 = df_train[["text", "climate"]]
df_train_model1.rename(columns={"climate": "label"}, inplace = 1)
df_train_model1

/usr/local/lib/python3.9/dist-packages/pandas/core/frame.py:5039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(


,text,label
0,The accelerator programs have sub-portfolios o...,0
1,"Also by means of BNDES Finem, we offer credit ...",1
2,Climate change Climate change exposes UPM to v...,0
3,Several tools and methodologies aimed at asses...,0
4,We worked with the UK government to accelerate...,0
...,...,...
395,"At the beginning of 2019, VINCI Airports signe...",1
396,We have also signed up to the Partnership for ...,0
397,Suzano also is involved and spearheads externa...,0
398,Risks to the Group’s reputation Risks include ...,1


In [82]:
df_val_model1 = df_val[["text", "climate"]]
df_val_model1.rename(columns={"climate": "label"}, inplace = 1)
df_val_model1

/usr/local/lib/python3.9/dist-packages/pandas/core/frame.py:5039: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().rename(


,text,label
0,"Except as required by law, the Bank does not u...",1
1,"In terms of thermal coal, the Group has set an...",0
2,The engagement is driving improved conversatio...,0
3,The tool does this by tracking the generation ...,0
4,We also support and advise companies in the La...,0
...,...,...
395,"JetBlue drives safety management in many ways,...",1
396,10 Representative Concentration Pathways (RCP)...,0
397,Building on our history of energy efficiency i...,0
398,"In 2019, PME redesigned the passively managed ...",0


In [83]:
train_dataset = datasets.Dataset.from_pandas(df_train_model1)
val_dataset = datasets.Dataset.from_pandas(df_val_model1)
val_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 400
})

In [84]:
train_texts, train_climate_labels = df_train_model1["text"], df_train_model1["label"]

In [85]:
print(tokenize({'text': train_dataset["text"][0], 'label': train_dataset["label"][0]}))

{'input_ids': [0, 133, 31463, 1767, 33, 2849, 12, 3427, 47039, 9, 2147, 12, 2061, 8666, 2800, 30, 1378, 10448, 18, 4470, 1942, 2567, 6, 217, 5, 11348, 154, 1628, 11001, 7229, 26514, 6, 8, 5, 33515, 13465, 25353, 9356, 359, 11001, 7229, 21273, 2176, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'label': 0}


In [86]:
train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize, remove_columns=["text"])
train_dataset

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 400
})

In [87]:
set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

training_args = TrainingArguments(
    output_dir="/content"
)

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

trainer.train()

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=150, training_loss=0.14864786783854167, metrics={'train_runtime': 23.4184, 'train_samples_per_second': 51.242, 'train_steps_per_second': 6.405, 'total_flos': 57615039202944.0, 'train_loss': 0.14864786783854167, 'epoch': 3.0})

In [88]:
pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1)))

***** Running Prediction *****
  Num examples = 400
  Batch size = 8


0.9153439153439153
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       329
           1       0.91      0.82      0.86        71

    accuracy                           0.95       400
   macro avg       0.93      0.90      0.92       400
weighted avg       0.95      0.95      0.95       400



In [15]:
# Tuning per_device_train_batch_size, per_device_eval_batch_size, num_train_epochs

maxa = 0
goodi = 0
goodj = 0

for i in range(5): 
  for j in range(5):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
    model = model.to(device)


    training_args = TrainingArguments(
    output_dir="/content",
    evaluation_strategy="epoch",
    per_device_train_batch_size=i + 5,
    per_device_eval_batch_size=i + 5,
    learning_rate=5e-5,
    num_train_epochs=j + 1
    )

    trainer = Trainer(
      model,
      training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      tokenizer=tokenizer,
  )

    trainer.train()
    pred = trainer.predict(val_dataset)

    if f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro') > maxa:
      maxa = f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro')
      goodi = i + 5
      goodj = j + 1                     
    print(i + 5, j + 1)
    print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.187235


***** Running Evaluation *****
  Num examples = 400
  Batch size = 5


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 400
  Batch size = 5


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

5 1
0.9170950091195491
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       329
           1       0.83      0.90      0.86        71

    accuracy                           0.95       400
   macro avg       0.90      0.93      0.92       400
weighted avg       0.95      0.95      0.95       400



...

9 4
0.9153116531165311
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       329
           1       0.85      0.87      0.86        71

    accuracy                           0.95       400
   macro avg       0.91      0.92      0.92       400
weighted avg       0.95      0.95      0.95       400



Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.decoder.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.weight'

Epoch,Training Loss,Validation Loss
1,No log,0.228174
2,No log,0.215607
3,No log,0.296592
4,No log,0.331054
5,No log,0.341173


***** Running Evaluation *****
  Num examples = 400
  Batch size = 9
***** Running Evaluation *****
  Num examples = 400
  Batch size = 9
***** Running Evaluation *****
  Num examples = 400
  Batch size = 9
***** Running Evaluation *****
  Num examples = 400
  Batch size = 9
***** Running Evaluation *****
  Num examples = 400
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 400
  Batch size = 9


9 5
0.903673540873068
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       329
           1       0.87      0.82      0.84        71

    accuracy                           0.94       400
   macro avg       0.91      0.89      0.90       400
weighted avg       0.94      0.94      0.94       400



In [16]:
# rerun code after tuning

set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

training_args = TrainingArguments(
output_dir="/content",
evaluation_strategy="epoch",
per_device_train_batch_size=goodi,
per_device_eval_batch_size=goodi,
#learning_rate=5e-5,
num_train_epochs=goodj
)

trainer = Trainer(
  model,
  training_args,
  train_dataset=train_dataset,
  eval_dataset=val_dataset,
  tokenizer=tokenizer,
)
trainer.train()

pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.214008
2,No log,0.269022
3,No log,0.380088
4,No log,0.288832


***** Running Evaluation *****
  Num examples = 400
  Batch size = 5
***** Running Evaluation *****
  Num examples = 400
  Batch size = 5
***** Running Evaluation *****
  Num examples = 400
  Batch size = 5
***** Running Evaluation *****
  Num examples = 400
  Batch size = 5


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 400
  Batch size = 5


0.9354276305169018
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       329
           1       0.90      0.89      0.89        71

    accuracy                           0.96       400
   macro avg       0.94      0.93      0.94       400
weighted avg       0.96      0.96      0.96       400



# *1st approach, 2nd model*


In [17]:
df_train_model2 = df_train[["text", "sentiment"]].dropna().reset_index(drop=True)
df_train_model2.rename(columns={"sentiment": "label"}, inplace = 1)
df_train_model2["label"] = df_train_model2["label"].astype(int)

df_train_model2

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,2
2,Several tools and methodologies aimed at asses...,2
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",2
...,...,...
327,The main impacts of the 4°C scenario were as f...,2
328,Emission sources not reported This section of ...,1
329,We have also signed up to the Partnership for ...,2
330,Suzano also is involved and spearheads externa...,1


In [18]:
df_val_model2 = df_val[["text", "sentiment"]].dropna().reset_index(drop=True)
df_val_model2.rename(columns={"sentiment": "label"}, inplace = 1)
df_val_model2["label"] = df_val_model2["label"].astype(int)

df_val_model2

,text,label
0,"In terms of thermal coal, the Group has set an...",1
1,The engagement is driving improved conversatio...,1
2,The tool does this by tracking the generation ...,1
3,We also support and advise companies in the La...,0
4,To control climate risks in insurance products...,2
...,...,...
324,Investec group including Investec Asset Manage...,2
325,10 Representative Concentration Pathways (RCP)...,1
326,Building on our history of energy efficiency i...,1
327,"In 2019, PME redesigned the passively managed ...",2


In [19]:
train_dataset = datasets.Dataset.from_pandas(df_train_model2)
val_dataset = datasets.Dataset.from_pandas(df_val_model2)
train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 332
})

In [20]:
train_texts, train_climate_labels = df_train_model2["text"], df_train_model2["label"]

In [21]:
train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize, remove_columns=["text"])
train_dataset

Map:   0%|          | 0/332 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 332
})

In [22]:

set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)  
model = model.to(device)

training_args = TrainingArguments(
    output_dir="/content"
)

trainer = Trainer(
    model,
    training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

trainer.train()

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265


Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=126, training_loss=0.5736101544092572, metrics={'train_runtime': 19.3532, 'train_samples_per_second': 51.464, 'train_steps_per_second': 6.511, 'total_flos': 49704103224216.0, 'train_loss': 0.5736101544092572, 'epoch': 3.0})

In [23]:
pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1)))

***** Running Prediction *****
  Num examples = 329
  Batch size = 8


0.8300573300573301
              precision    recall  f1-score   support

           0       0.76      0.81      0.79        81
           1       0.84      0.76      0.80       136
           2       0.88      0.94      0.91       112

    accuracy                           0.83       329
   macro avg       0.83      0.84      0.83       329
weighted avg       0.83      0.83      0.83       329



In [24]:
maxa = 0
goodi = 0
goodj = 0
for i in range(5): 
  for j in range(5):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)  
    model = model.to(device)


    training_args = TrainingArguments(
    output_dir="/content",
    evaluation_strategy="epoch",
    per_device_train_batch_size=i + 5,
    per_device_eval_batch_size=i + 5,
    #learning_rate=5e-5,
    num_train_epochs=j + 1
    )

    trainer = Trainer(
      model,
      training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      tokenizer=tokenizer,
  )

    trainer.train()
    pred = trainer.predict(val_dataset)

    if f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro') > maxa:
      maxa = f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro')
      goodi = i + 5
      goodj = j + 1                     
    print(i + 5, j + 1)
    print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265


Epoch,Training Loss,Validation Loss
1,No log,0.648276


***** Running Evaluation *****
  Num examples = 329
  Batch size = 5


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 5


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265


5 1
0.7581323796783525
              precision    recall  f1-score   support

           0       0.67      0.74      0.70        81
           1       0.70      0.76      0.73       136
           2       0.95      0.77      0.85       112

    accuracy                           0.76       329
   macro avg       0.77      0.76      0.76       329
weighted avg       0.77      0.76      0.76       329



...

9 4
0.8255393040806785
              precision    recall  f1-score   support

           0       0.79      0.78      0.78        81
           1       0.78      0.82      0.80       136
           2       0.92      0.87      0.89       112

    accuracy                           0.83       329
   macro avg       0.83      0.82      0.83       329
weighted avg       0.83      0.83      0.83       329



Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.decoder.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.weight'

Epoch,Training Loss,Validation Loss
1,No log,0.649284
2,No log,0.471255
3,No log,0.676307
4,No log,0.816340
5,No log,0.792605


***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 9


9 5
0.8051720359412667
              precision    recall  f1-score   support

           0       0.73      0.79      0.76        81
           1       0.77      0.77      0.77       136
           2       0.92      0.86      0.89       112

    accuracy                           0.81       329
   macro avg       0.81      0.81      0.81       329
weighted avg       0.81      0.81      0.81       329



In [25]:
print(maxa, goodi, goodj)

0.8354714219176259 8 4


In [26]:
set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3)  
model = model.to(device)

training_args = TrainingArguments(
output_dir="/content",
evaluation_strategy="epoch",
per_device_train_batch_size=goodi,
per_device_eval_batch_size=goodi,
#learning_rate=5e-5,
num_train_epochs=goodj
)

trainer = Trainer(
  model,
  training_args,
  train_dataset=train_dataset,
  eval_dataset=val_dataset,
  tokenizer=tokenizer,
)
trainer.train()

pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265


Epoch,Training Loss,Validation Loss
1,No log,0.712257
2,No log,0.459054
3,No log,0.609257
4,No log,0.523342


***** Running Evaluation *****
  Num examples = 329
  Batch size = 8
***** Running Evaluation *****
  Num examples = 329
  Batch size = 8
***** Running Evaluation *****
  Num examples = 329
  Batch size = 8
***** Running Evaluation *****
  Num examples = 329
  Batch size = 8


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 8


0.8354714219176259
              precision    recall  f1-score   support

           0       0.85      0.74      0.79        81
           1       0.79      0.84      0.81       136
           2       0.89      0.91      0.90       112

    accuracy                           0.84       329
   macro avg       0.84      0.83      0.84       329
weighted avg       0.84      0.84      0.84       329



# *1st approach, 3rd model*



In [27]:
df_train_model3 = df_train[["text", "commitment"]].dropna().reset_index(drop=True)
df_train_model3.rename(columns={"commitment": "label"}, inplace = 1)
df_train_model3["label"] = df_train_model3["label"].astype(int)
df_train_model3

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,1
2,Several tools and methodologies aimed at asses...,0
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",1
...,...,...
327,The main impacts of the 4°C scenario were as f...,1
328,Emission sources not reported This section of ...,1
329,We have also signed up to the Partnership for ...,0
330,Suzano also is involved and spearheads externa...,0


In [28]:
df_val_model3 = df_val[["text", "commitment"]].dropna().reset_index(drop=True)
df_val_model3.rename(columns={"commitment": "label"}, inplace = 1)
df_val_model3["label"] = df_val_model3["label"].astype(int)
df_val_model3

,text,label
0,"In terms of thermal coal, the Group has set an...",0
1,The engagement is driving improved conversatio...,0
2,The tool does this by tracking the generation ...,1
3,We also support and advise companies in the La...,0
4,To control climate risks in insurance products...,1
...,...,...
324,Investec group including Investec Asset Manage...,1
325,10 Representative Concentration Pathways (RCP)...,1
326,Building on our history of energy efficiency i...,0
327,"In 2019, PME redesigned the passively managed ...",0


In [29]:
train_dataset = datasets.Dataset.from_pandas(df_train_model3)
val_dataset = datasets.Dataset.from_pandas(df_val_model3)
val_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 329
})

In [30]:
train_texts, train_climate_labels = df_train_model3["text"], df_train_model3["label"]

In [31]:
train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize, remove_columns=["text"])
train_dataset

Map:   0%|          | 0/332 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 332
})

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

In [33]:
trainer = Trainer(
    model,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

trainer.train()

No `TrainingArguments` passed, using `output_dir=tmp_trainer`.
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
/usr/local/lib/python3.9/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 332
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 126
  Number of trainable parameters = 82119938


Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=126, training_loss=0.3541896456763858, metrics={'train_runtime': 19.3701, 'train_samples_per_second': 51.419, 'train_steps_per_second': 6.505, 'total_flos': 49703216837904.0, 'train_loss': 0.3541896456763858, 'epoch': 3.0})

In [34]:
pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1)))

***** Running Prediction *****
  Num examples = 329
  Batch size = 8


0.8978693382249189
              precision    recall  f1-score   support

           0       0.86      0.91      0.88       139
           1       0.93      0.89      0.91       190

    accuracy                           0.90       329
   macro avg       0.90      0.90      0.90       329
weighted avg       0.90      0.90      0.90       329



In [35]:
maxa = 0
goodi = 0
goodj = 0
for i in range(5): 
  for j in range(5):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
    model = model.to(device)


    training_args = TrainingArguments(
    output_dir="/content",
    evaluation_strategy="epoch",
    per_device_train_batch_size=i + 5,
    per_device_eval_batch_size=i + 5,
    #learning_rate=5e-5,
    num_train_epochs=j + 1
    )

    trainer = Trainer(
      model,
      training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      tokenizer=tokenizer,
  )

    trainer.train()
    pred = trainer.predict(val_dataset)

    if f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro') > maxa:
      maxa = f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro')
      goodi = i + 5
      goodj = j + 1                     
    print(i + 5, j + 1)
    print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    # print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.399502


***** Running Evaluation *****
  Num examples = 329
  Batch size = 5


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 5


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

5 1
0.8472478846281399


...

9 4
0.8983398410067698


Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.decoder.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.weight'

Epoch,Training Loss,Validation Loss
1,No log,0.478066
2,No log,0.284664
3,No log,0.487349
4,No log,0.467856
5,No log,0.565771


***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 9


9 5
0.8954735740450026


In [36]:
print(maxa, goodi, goodj)

0.9042140556176452 9 3


In [37]:
set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

training_args = TrainingArguments(
output_dir="/content",
evaluation_strategy="epoch",
per_device_train_batch_size=goodi,
per_device_eval_batch_size=goodi,
num_train_epochs=goodj
)

trainer = Trainer(
  model,
  training_args,
  train_dataset=train_dataset,
  eval_dataset=val_dataset,
  tokenizer=tokenizer,
)
trainer.train()

pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.447446
2,No log,0.285480
3,No log,0.328311


***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 9


0.9042140556176452
              precision    recall  f1-score   support

           0       0.86      0.92      0.89       139
           1       0.94      0.89      0.92       190

    accuracy                           0.91       329
   macro avg       0.90      0.91      0.90       329
weighted avg       0.91      0.91      0.91       329



# *1st approach, 4th model*



In [38]:
df_train_model4 = df_train[["text", "specificity"]].dropna().reset_index(drop=True)
df_train_model4.rename(columns={"specificity": "label"}, inplace = 1)
df_train_model4 = df_train_model4.dropna()
df_train_model4["label"] = df_train_model4["label"].astype(int)
df_train_model4

,text,label
0,The accelerator programs have sub-portfolios o...,1
1,Climate change Climate change exposes UPM to v...,1
2,Several tools and methodologies aimed at asses...,1
3,We worked with the UK government to accelerate...,0
4,"(changes in consumption behavior, failure of i...",1
...,...,...
327,The main impacts of the 4°C scenario were as f...,1
328,Emission sources not reported This section of ...,0
329,We have also signed up to the Partnership for ...,1
330,Suzano also is involved and spearheads externa...,1


In [39]:
df_val_model4 = df_val[["text", "specificity"]].dropna().reset_index(drop=True)
df_val_model4.rename(columns={"specificity": "label"}, inplace = 1)
df_val_model4 = df_val_model4.dropna()
df_val_model4["label"] = df_val_model4["label"].astype(int)
df_val_model4

,text,label
0,"In terms of thermal coal, the Group has set an...",0
1,The engagement is driving improved conversatio...,1
2,The tool does this by tracking the generation ...,1
3,We also support and advise companies in the La...,0
4,To control climate risks in insurance products...,1
...,...,...
324,Investec group including Investec Asset Manage...,1
325,10 Representative Concentration Pathways (RCP)...,1
326,Building on our history of energy efficiency i...,0
327,"In 2019, PME redesigned the passively managed ...",0


In [40]:
train_dataset = datasets.Dataset.from_pandas(df_train_model3)
val_dataset = datasets.Dataset.from_pandas(df_val_model3)
val_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 329
})

In [41]:
train_texts, train_climate_labels = df_train_model4["text"], df_train_model4["label"]

In [42]:
train_dataset = train_dataset.map(tokenize, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize, remove_columns=["text"])
train_dataset

Map:   0%|          | 0/332 [00:00<?, ? examples/s]

Map:   0%|          | 0/329 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 332
})

In [43]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

In [44]:
trainer = Trainer(
    model,
    train_dataset=train_dataset,
    tokenizer=tokenizer
)

trainer.train()

No `TrainingArguments` passed, using `output_dir=tmp_trainer`.
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
/usr/local/lib/python3.9/dist-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 332
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 126
  Number of trainable parameters = 82119938


Step,Training Loss




Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=126, training_loss=0.3148329598563058, metrics={'train_runtime': 19.3594, 'train_samples_per_second': 51.448, 'train_steps_per_second': 6.508, 'total_flos': 49703216837904.0, 'train_loss': 0.3148329598563058, 'epoch': 3.0})

In [45]:
pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

***** Running Prediction *****
  Num examples = 329
  Batch size = 8


0.9008812231678843
              precision    recall  f1-score   support

           0       0.87      0.91      0.89       139
           1       0.93      0.90      0.91       190

    accuracy                           0.90       329
   macro avg       0.90      0.90      0.90       329
weighted avg       0.90      0.90      0.90       329



In [46]:
maxa = 0
goodi = 0
goodj = 0
for i in range(5): 
  for j in range(5):
    set_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
    model = model.to(device)


    training_args = TrainingArguments(
    output_dir="/content",
    evaluation_strategy="epoch",
    per_device_train_batch_size=i + 5,
    per_device_eval_batch_size=i + 5,
    #learning_rate=5e-5,
    num_train_epochs=j + 1
    )

    trainer = Trainer(
      model,
      training_args,
      train_dataset=train_dataset,
      eval_dataset=val_dataset,
      tokenizer=tokenizer,
  )

    trainer.train()
    pred = trainer.predict(val_dataset)

    if f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro') > maxa:
      maxa = f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro')
      goodi = i + 5
      goodj = j + 1                     
    print(i + 5, j + 1)
    print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
    # print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.399502


***** Running Evaluation *****
  Num examples = 329
  Batch size = 5


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 5


loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

5 1
0.8472478846281399


...

9 4
0.8983398410067698


Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForSequenceClassification: ['lm_head.layer_norm.bias', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.decoder.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.weight'

Epoch,Training Loss,Validation Loss
1,No log,0.478066
2,No log,0.284664
3,No log,0.487349
4,No log,0.467856
5,No log,0.565771


***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 9


9 5
0.8954735740450026


In [47]:
print(maxa, goodi, goodj)

0.9042140556176452 9 3


In [48]:
set_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)  
model = model.to(device)

training_args = TrainingArguments(
output_dir="/content",
evaluation_strategy="epoch",
per_device_train_batch_size=goodi,
per_device_eval_batch_size=goodi,
#learning_rate=5e-5,
num_train_epochs=goodj
)

trainer = Trainer(
  model,
  training_args,
  train_dataset=train_dataset,
  eval_dataset=val_dataset,
  tokenizer=tokenizer,
)
trainer.train()

pred = trainer.predict(val_dataset)
print(f1_score(pred.label_ids, pred.predictions.argmax(1), average='macro'))
print(classification_report(pred.label_ids,pred.predictions.argmax(1))) 

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a4c10df32/config.json
Model config RobertaConfig {
  "_name_or_path": "distilroberta-base",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.26.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--distilroberta-base/snapshots/d5411c3ee9e1793fd9ef58390b40a80a

Epoch,Training Loss,Validation Loss
1,No log,0.447446
2,No log,0.285480
3,No log,0.328311


***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9
***** Running Evaluation *****
  Num examples = 329
  Batch size = 9


Training completed. Do not forget to share your model on huggingface.co/models =)


***** Running Prediction *****
  Num examples = 329
  Batch size = 9


0.9042140556176452
              precision    recall  f1-score   support

           0       0.86      0.92      0.89       139
           1       0.94      0.89      0.92       190

    accuracy                           0.91       329
   macro avg       0.90      0.91      0.90       329
weighted avg       0.91      0.91      0.91       329

